In [2]:
import pandas as pd
import numpy as np
import os
from xgboost import XGBRegressor
from tqdm import tqdm
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Suppress warnings
warnings.filterwarnings('ignore')

# Load the feature-engineered data
print("Loading data...")
df = pd.read_parquet("full_dtime_feature_engineered_label_encoded.parquet")

# Define province and commodity mappings
provinces = ['Aceh', 'Bali', 'Banten', 'Bengkulu', 'DI Yogyakarta',
    'DKI Jakarta', 'Gorontalo', 'Jambi', 'Jawa Barat', 'Jawa Tengah',
    'Jawa Timur', 'Kalimantan Barat', 'Kalimantan Selatan',
    'Kalimantan Tengah', 'Kalimantan Timur', 'Kalimantan Utara',
    'Kepulauan Bangka Belitung', 'Kepulauan Riau', 'Lampung', 'Maluku',
    'Maluku Utara', 'Nusa Tenggara Barat', 'Nusa Tenggara Timur',
    'Papua', 'Papua Barat', 'Riau', 'Sulawesi Barat',
    'Sulawesi Selatan', 'Sulawesi Tengah', 'Sulawesi Tenggara',
    'Sulawesi Utara', 'Sumatera Barat', 'Sumatera Selatan',
    'Sumatera Utara']

commodities = ['Bawang Merah', 'Bawang Putih Bonggol', 'Beras Medium',
    'Beras Premium', 'Cabai Merah Keriting', 'Cabai Rawit Merah',
    'Daging Ayam Ras', 'Daging Sapi Murni', 'Gula Konsumsi',
    'Minyak Goreng Curah', 'Minyak Goreng Kemasan Sederhana',
    'Telur Ayam Ras', 'Tepung Terigu (Curah)']

# Create directory for models and output if they don't exist
os.makedirs("models", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Generate date features for the prediction period
print("Generating prediction dates...")
pred_dates = pd.date_range(start="2024-10-01", end="2024-12-31", freq="D")
dfs = pd.DataFrame(pred_dates, columns=["date"])

# Create time features
dfs['dayofweek'] = dfs['date'].dt.dayofweek
dfs['day'] = dfs['date'].dt.day
dfs['month'] = dfs['date'].dt.month
dfs['year'] = dfs['date'].dt.year
dfs['quarter'] = dfs['date'].dt.quarter
dfs['weekofyear'] = dfs['date'].dt.isocalendar().week
dfs['sin_month'] = np.sin(2 * np.pi * dfs['month'] / 12)
dfs['cos_month'] = np.cos(2 * np.pi * dfs['month'] / 12)
dfs['sin_day'] = np.sin(2 * np.pi * dfs['day'] / 31)
dfs['cos_day'] = np.cos(2 * np.pi * dfs['day'] / 31)
dfs['sin_dayofweek'] = np.sin(2 * np.pi * dfs['dayofweek'] / 7)
dfs['cos_dayofweek'] = np.cos(2 * np.pi * dfs['dayofweek'] / 7)
dfs['sin_weekofyear'] = np.sin(2 * np.pi * dfs['weekofyear'] / 52)
dfs['cos_weekofyear'] = np.cos(2 * np.pi * dfs['weekofyear'] / 52)
dfs['sin_quarter'] = np.sin(2 * np.pi * dfs['quarter'] / 4)
dfs['cos_quarter'] = np.cos(2 * np.pi * dfs['quarter'] / 4)

# Define the feature columns
features = ['dayofweek', 'day', 'month', 'year', 'quarter', 'weekofyear', 
         'sin_month', 'cos_month', 'sin_day', 'cos_day', 
         'sin_dayofweek', 'cos_dayofweek', 'sin_weekofyear', 
         'cos_weekofyear', 'sin_quarter', 'cos_quarter']

# Store all predictions
all_predictions = []
# Store all model evaluations
model_evaluations = []

# Function to calculate MAPE
def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Loop through each commodity and province combination
for comm_idx, commodity in enumerate(tqdm(range(len(commodities)), desc="Commodities")):
    for prov_idx, province in enumerate(tqdm(range(len(provinces)), desc=f"Provinces for {commodities[comm_idx]}")):
     try:
         # Filter the data for this commodity-province pair
         filtered_df = df[(df["Comodity"] == comm_idx) & (df["Provinsi"] == prov_idx)]
         
         if len(filtered_df) == 0:
          print(f"No data for {commodities[comm_idx]}/{provinces[prov_idx]}, skipping...")
          continue
          
         # Prepare features for training
         X = filtered_df[features]
         y = filtered_df["Price"]
         
         # Split data into training and testing sets (80% train, 20% test)
         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
         
         # Train an XGBoost model
         model = XGBRegressor(
          n_estimators=1000, 
          max_depth=10, 
          learning_rate=0.2, 
          n_jobs=-1,
          objective="reg:squarederror",
          early_stopping_rounds=50,
          verbosity=0
         )
         
         # Train the model with validation
         model.fit(
          X_train, 
          y_train, 
          eval_set=[(X_test, y_test)],
          verbose=0
         )
         
         # Evaluate model on test set
         y_pred = model.predict(X_test)
         mape = calculate_mape(y_test, y_pred)
         mae = mean_absolute_error(y_test, y_pred)
         
         # Store evaluation metrics
         model_evaluations.append({
          'commodity': commodities[comm_idx],
          'province': provinces[prov_idx],
          'MAPE': mape,
          'MAE': mae,
          'samples': len(filtered_df)
         })
         
         print(f"Model for {commodities[comm_idx]}/{provinces[prov_idx]} - MAPE: {mape:.2f}%, MAE: {mae:.2f}")
         
         # Save the model
         model.save_model(f"models/{commodities[comm_idx]}_{provinces[prov_idx]}.json")
         
         # Prepare prediction features
         pred_features = dfs[features].copy()
         
         # Make predictions
         predictions = model.predict(pred_features)
         
         # Create DataFrame with results
         result_df = pd.DataFrame({
          'date': pred_dates,
          'commodity': commodities[comm_idx],
          'province': provinces[prov_idx],
          'price': predictions
         })
         
         # Format the id column
         result_df['id'] = result_df.apply(
          lambda row: f"{row['commodity']}/{row['province']}/{row['date'].strftime('%Y-%m-%d')}", 
          axis=1
         )
         
         # Select only id and price columns
         result_df = result_df[['id', 'price']]
         
         # Save individual predictions
         result_df.to_csv(f"output/{commodities[comm_idx]}_{provinces[prov_idx]}.csv", index=False)
         
         # Append to all predictions
         all_predictions.append(result_df)
          
     except Exception as e:
         print(f"Error processing {commodities[comm_idx]}/{provinces[prov_idx]}: {str(e)}")

# Save model evaluation results
eval_df = pd.DataFrame(model_evaluations)
eval_df.to_csv("model_evaluations.csv", index=False)
print(f"Average MAPE across all models: {eval_df['MAPE'].mean():.2f}%")

# Combine all predictions
print("Combining all predictions...")
final_predictions = pd.concat(all_predictions, ignore_index=True)

# Save the final CSV
final_predictions.to_csv("commodity_price_predictions.csv", index=False)

print("Forecasting complete! Results saved to commodity_price_predictions.csv")

Loading data...
Generating prediction dates...


Commodities:   0%|          | 0/13 [00:00<?, ?it/s]


Model for Bawang Merah/Aceh - MAPE: 1.42%, MAE: 510.96


Provinces for Bawang Merah:   3%|▎         | 1/34 [00:00<00:27,  1.19it/s]

Model for Bawang Merah/Bali - MAPE: 1.67%, MAE: 484.37
Model for Bawang Merah/Banten - MAPE: 2.51%, MAE: 921.04


...

Model for Tepung Terigu (Curah)/Sumatera Barat - MAPE: 0.87%, MAE: 92.30


Model for Tepung Terigu (Curah)/Sumatera Selatan - MAPE: 0.41%, MAE: 40.08


Commodities: 100%|██████████| 13/13 [06:30<00:00, 30.08s/it]

Model for Tepung Terigu (Curah)/Sumatera Utara - MAPE: 0.64%, MAE: 67.40
Average MAPE across all models: 1.38%
Combining all predictions...


Forecasting complete! Results saved to commodity_price_predictions.csv
